# Lesson 10: AI System Architecture and Continuous Evolution — From Prototype to Production

## Learning Objectives
- Understand where guardrails sit inside the production five-layer architecture
>
> The full five layers (context enhancement / guardrails / router & gateway / cache / agent patterns) are covered in the concept track — see Lesson 10 of the course guide.
- Implement simple input guardrails and output filtering
- Design a user feedback collection mechanism
- Complete a comprehensive review of the core knowledge across all ten lessons

> Connect everything you've learned in the previous nine lessons and understand how a real AI system actually operates.

## Environment Setup

> Please run `00_Environment_Setup.ipynb` first to set up dependencies and API keys,
> then return to this notebook.

Once done, run the cell below to load environment variables:

In [ ]:
# Load API key from .env file (no need to enter it every time)
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== Pick a provider: change this one line, nothing else =====
#   'openai'     cloud  needs OPENAI_API_KEY      strongest, has embeddings
#   'deepseek'   cloud  needs DEEPSEEK_API_KEY    cheapest cloud, no embeddings
#   'openrouter' cloud  needs OPENROUTER_API_KEY  many vendors, no embeddings
#   'ollama'     local  no key, free and offline  run `ollama serve` and pull the model first
PROVIDER = 'openai'

# All four speak the OpenAI API format. They differ only in URL, key, model names.
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = OpenAI's default endpoint
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # small model: cheap and fast
        'model_big': 'gpt-5.6-terra',                # big model: pricier and stronger
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # fast and cheap
        'model_big': 'deepseek-v4-pro',              # stronger and slower; both V4 models think first
        'embedding_model': None,                     # DeepSeek has no embeddings endpoint
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter does not proxy embeddings
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # local models ignore the key
        'model': 'gemma4:e2b-mlx',                   # run: ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # run: ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# Check the key first: with no key, the OpenAI client raises a long traceback.
if not cfg['api_key']:
    raise SystemExit(
        f"No API key for '{PROVIDER}'. Either add {PROVIDER.upper()}_API_KEY to your .env file,\n"
        f"or set PROVIDER = 'ollama' above to run locally with no key at all."
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# Every lesson below uses only these three names, so switching provider needs no
# other code change.
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'Connected! provider = {PROVIDER}, default model = {MODEL}')


---

## Activity 1: Build "Guardrails" for Your AI System

### Activity Goal
Guardrails are the safety lines of an AI system. They include input guardrails (filter harmful inputs) and output guardrails (filter inappropriate outputs).
This activity lets you implement simple guardrail mechanisms in code.

In [ ]:
# Activity 1: Implement input guardrails

# Define sensitive keyword list
sensitive_keywords = ['password', 'credit card number', 'SSN', 'social security', 'home address', 'violence', 'suicide', 'nude']

def input_guardrail(user_input):
    """Input guardrail: detect and block inappropriate inputs"""
    hits = []
    for keyword in sensitive_keywords:
        if keyword in user_input.lower():
            hits.append(keyword)
    if hits:
        return False, f'Input contains sensitive keywords: {", ".join(hits)}. Blocked.'
    return True, 'Safe'

# Test input guardrail
test_inputs = [
    'What is the weather like today?',
    'Can you help me find my credit card number?',
    'Recommend a good book about AI.',
    'How to solve problems with violence?',
]

print('=== Input Guardrail Test ===')
for inp in test_inputs:
    ok, msg = input_guardrail(inp)
    status = 'PASS' if ok else 'BLOCKED'
    print(f'[{status}] {inp}')
    if not ok:
        print(f'  Reason: {msg}')

### Output Guardrails

In [ ]:
# Output guardrail: filter inappropriate AI output

def output_guardrail(ai_response):
    """Output guardrail: detect risky content in AI responses"""
    warnings = []
    # Check if investment advice is given without a disclaimer
    if 'invest' in ai_response.lower() and 'risk' not in ai_response.lower():
        warnings.append('Missing risk disclaimer')
    # Check if AI is "playing doctor"
    medical_keywords = ['diagnosis', 'prescription', 'medication advice']
    for kw in medical_keywords:
        if kw in ai_response.lower():
            warnings.append(f'Involves medical advice ({kw}). Should include "for reference only, consult a doctor"')
    return warnings

# Test output guardrail
print('=== Output Guardrail Test ===')

# Simulate a potentially risky AI response
risky_response = 'Based on your symptoms, my diagnosis is a mild infection. My prescription: take aspirin twice daily.'
print(f'AI Response: {risky_response}')
warnings = output_guardrail(risky_response)
if warnings:
    print('Warnings:')
    for w in warnings:
        print(f'  - {w}')
else:
    print('Output check passed. No risk detected.')

# Simulate a safe response
safe_response = 'Regarding your health concern, please consult a doctor. The following is for reference only: ...'
print(f'\nAI Response: {safe_response}')
warnings = output_guardrail(safe_response)
if warnings:
    print('Warnings:')
    for w in warnings:
        print(f'  - {w}')
else:
    print('Output check passed. No risk detected.')

### Discussion
- Which is harder to implement: input guardrails or output guardrails? Why?
- Guardrails that are too strict limit AI capability; too loose creates security risks — how do you find the balance?
- What if a user tries to bypass guardrails using metaphors or roundabout language?

---

## Activity 2: Design a User Feedback Collection System

### Activity Goal
User feedback is the fuel for continuous AI evolution. Designing a good feedback collection mechanism is key to moving AI from "it works" to "it works well."
Feedback is not about sending out surveys — it's about naturally capturing signals within the conversational flow.

In [ ]:
# Activity 2: Simulate a feedback collection system

# Simulate a conversation history
conversation = [
    {'role': 'user', 'content': 'When will my order ship?'},
    {'role': 'assistant', 'content': 'Your order #12345 is expected to ship tomorrow. The tracking number will be sent via text message.'},
    {'role': 'user', 'content': 'Great, thank you!'},
    {'role': 'assistant', 'content': 'You are welcome! Is there anything else I can help with?'},
]

# Feedback collection: automatically analyze conversation after it ends
def analyze_conversation_feedback(conversation):
    """Automatically extract feedback signals from the conversation"""
    signals = {
        'user_said_thanks': False,
        'user_asked_followup': False,
        'conversation_length': len(conversation),
        'resolution_likely': False
    }
    for idx, msg in enumerate(conversation):
        if msg['role'] == 'user':
            if any(w in msg['content'].lower() for w in ['thank', 'thanks', 'great', 'solved', 'perfect']):
                signals['user_said_thanks'] = True
                signals['resolution_likely'] = True
            # The opening question is not a follow-up: it only counts once the
            # assistant has already answered at least one turn.
            if idx > 0 and '?' in msg['content']:
                signals['user_asked_followup'] = True
    return signals

signals = analyze_conversation_feedback(conversation)

print('=== Automatic Feedback Analysis ===')
print(f'Conversation turns: {signals["conversation_length"]}')
print(f'User said thanks: {signals["user_said_thanks"]}')
print(f'User asked follow-up: {signals["user_asked_followup"]}')
print(f'Likely resolved: {signals["resolution_likely"]}')

# Feedback summary
print('\n=== Feedback Summary ===')
summary = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'You analyze customer service conversations and assess user satisfaction.'},
        {'role':'user','content':f'Analyze the following customer service conversation. Is the user satisfied? Give your reasoning:\n{conversation}'}
    ],
    temperature=0.2)
print(summary.choices[0].message.content)

print('\nTypes of Feedback Signals:')
print('  Explicit feedback: thumbs up/down, ratings, comments → direct but low collection rate')
print('  Implicit feedback: copied answer, closed page, said thanks, asked follow-up → indirect but high collection rate')
print('  Best practice: use both, with implicit as a safety net and explicit for validation')

### Discussion
- When you use AI yourself, do you give feedback? Why or why not?
- If users rarely provide explicit feedback, how can you still get meaningful signals?
- Is more feedback always better? How do you avoid "feedback fatigue"?

---

## Activity 3: Capstone Project — Connecting All Ten Lessons

### Activity Goal
Apply everything you've learned to build a small end-to-end project: from input guardrail → model inference → output evaluation → feedback collection.

In [ ]:
# Activity 3: Capstone Project — End-to-End AI Customer Service Pipeline

print('='*60)
print('Capstone Project: AI Customer Service Pipeline')
print('='*60)

# Simulate a user input
user_query = "I still haven't received my order. Are you a scam company? I want to file a complaint!"
print(f'User Input: {user_query}')

# Step 1: Input Guardrail (Lesson 10)
ok, msg = input_guardrail(user_query)
if not ok:
    print(f'[Input Guardrail] BLOCKED: {msg}')
else:
    print('[Input Guardrail] PASSED')

# Step 2: Intent classification + role setting (Lesson 5 — Prompt Engineering)
system_prompt = ('You are an empathetic customer service representative. '
                'Rules: 1) First, apologize and calm the customer down '
                '2) Provide a specific solution '
                '3) Maintain a warm, professional tone')

# Step 3: Call the model (Lessons 1-2)
print('[Model Inference] Processing...')
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':system_prompt},
        {'role':'user','content':user_query}
    ],
    temperature=0.3)  # Low temperature for consistent customer service (Lesson 2)
ai_response = r.choices[0].message.content

# Step 4: Output Guardrail (Lesson 10)
warnings = output_guardrail(ai_response)
if warnings:
    print('[Output Guardrail] Warnings:')
    for w in warnings:
        print(f'  - {w}')
else:
    print('[Output Guardrail] PASSED')

# Step 5: Output Evaluation (Lessons 3-4)
print('\n[AI Evaluation] Assessing response quality...')
eval_r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'You are a strict customer service quality evaluator.'},
        {'role':'user','content':f'Rate the following customer service response (Accuracy / Empathy / Resolution Ability, each 1-5):\n{ai_response}'}
    ],
    temperature=0.2)

# Final output
print('='*60)
print('Final AI Response:')
print('='*60)
print(ai_response)

print('='*60)
print('Quality Evaluation:')
print('='*60)
print(eval_r.choices[0].message.content)

print('\nThis pipeline integrates:')
print('  Lessons 1-2: Model invocation and parameter control')
print('  Lessons 3-4: AI evaluation')
print('  Lesson 5: Prompt engineering (system prompt design)')
print('  Lesson 10: Input/output guardrails')

---

## Review: The Complete AI Engineering Landscape

### Knowledge Map

| Lesson | Topic | Core Skill |
|--------|-------|------------|
| 1 | Intro to AI Engineering | API calls, role setting, capability boundaries |
| 2 | Understanding Foundation Models | Temperature/Top-P, hallucination detection |
| 3 | AI Evaluation Fundamentals | Manual evaluation, AI-as-judge, A/B comparison |
| 4 | AI System Evaluation in Practice | Batch evaluation, model selection, evaluation guides |
| 5 | Prompt Engineering | Prompt iteration, few-shot learning, safety boundaries |
| 6 | RAG and AI Agents | RAG systems, semantic search, agentic behavior |
| 7 | Model Fine-Tuning | Fine-tuning concepts, data preparation, method comparison |
| 8 | Data Is King | Dirty data detection, data synthesis, augmentation |
| 9 | Inference Optimization | Speed measurement, cost estimation, quantization |
| 10 | AI Architecture and Evolution | Guardrails, feedback systems, end-to-end pipelines |

### Core Insight

> The heart of AI Engineering is not the technology itself — it's how to turn technology into a means of solving problems.
> Evaluation matters more than development. Data is more critical than algorithms. User feedback is more precious than model parameters.

### Homework
1. Review all 10 notebooks and redo the activity that resonated with you the most
2. Design your own AI application plan: scenario, model choice, evaluation method, feedback mechanism
3. Continue learning: recommended resources include *AI Engineering*, OpenAI Cookbook, and LangChain documentation